# Stage 4 — CBAM Attention Augmentation

End-to-end runner mirroring `stage3/notebooks/stage3_colab.ipynb`.
Per-subset architecture is locked to the Stage 3 finalist (see
`stage3/docs/stage3_analysis.md`):

- FD001 → AttnRecurrent (GRU, **no GNN**)
- FD002 → AttnRecurrentGNN (BiGRU + GNN-physical)
- FD003 → AttnRecurrentGNN (GRU   + GNN-physical)
- FD004 → AttnRecurrentGNN (BiGRU + GNN-physical)

Stage 4 ablation matrix per subset: `{channel_only, temporal_only, cbam_full}`
× seeds `{7, 42, 123}` = 9 runs per subset, 36 runs total.

## Cell 1 — Clone or update the repo

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/m8wei-coder/ECE-228-project"
REPO_DIR = "/content/ECE-228-project"

if os.path.isdir(REPO_DIR):
    print(f"repo exists at {REPO_DIR}, pulling...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print(f"cloning {REPO_URL} -> {REPO_DIR} ...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!ls -la

## Cell 2 — Install dependencies

In [ ]:
# Stage 4 needs the same deps as Stage 3 (no extra libraries for CBAM).
!pip install -q torch_geometric
!pip install -q pyyaml joblib
print("deps installed")

## Cell 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/ece228_stage4"
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
print("Drive root:", DRIVE_ROOT)

## Cell 4 — Environment self-check

In [ ]:
import sys
if "/content/ECE-228-project" not in sys.path:
    sys.path.insert(0, "/content/ECE-228-project")

import torch
print("torch:           ", torch.__version__)
print("cuda available:  ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("  device:        ", torch.cuda.get_device_name(0))

import torch_geometric
print("torch_geometric: ", torch_geometric.__version__)

from stage2_refactor.training.trainer import fit
from stage3.models.recurrent_gnn import RecurrentGNNFusion
from stage4.models.attention import CBAM1D
from stage4.models.attn_recurrent import AttnRecurrent
from stage4.models.attn_recurrent_gnn import AttnRecurrentGNN
import stage4.train_stage4 as t4
print("stage2/stage3/stage4 imports OK")

# Quick CBAM param overhead summary.
for F in (14, 16, 21):
    print(f"  CBAM1D(F={F:>2}, r=4)  ->  {CBAM1D.parameter_count(F, 4)} params")

## Cell 5 — Build adjacency matrices (reuse Stage 3 builder)

Stage 4 uses the same `physical` graph that won Stage 3.

In [ ]:
import subprocess, sys
PEARSON_THRESHOLD = 0.3
result = subprocess.run(
    [sys.executable, "-m", "stage3.build_graph",
     "--pearson-threshold", str(PEARSON_THRESHOLD)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

import os
for f in sorted(os.listdir("stage3/artifacts")):
    if f.startswith("adj_") and f.endswith(".npy"):
        print("  ", f)

## Cell 6 — Single-run hyperparameters (the only knob cell)

Edit this cell to launch a single Stage 4 run from Cell 7.

In [ ]:
SUBSET           = "FD002"        # FD001 / FD002 / FD003 / FD004
USE_CHANNEL_ATTN = True
USE_TEMPORAL_ATTN = True
GRAPH_METHOD     = "physical"     # physical / pearson / union (FD002-4 only)
ATTN_REDUCTION   = 4
ATTN_KERNEL      = 7

# Backbone overrides (None = per-subset Stage 3 finalist default).
RECURRENT_KIND   = None
HIDDEN_SIZE      = None
NUM_LAYERS       = None
DROPOUT          = None
LR               = None
BATCH_SIZE       = None
EPOCHS           = None

# GNN branch hyperparams (ignored for FD001).
GNN_HIDDEN       = 32
GNN_LAYERS       = 2
GNN_KIND         = "gcn"
GNN_DROPOUT      = 0.1
GNN_POOL         = "mean"

SEED             = 42
RUN_ID           = None           # None -> auto from flags

## Cell 7 — Train a single Stage 4 run

In [ ]:
import sys, importlib
import stage4.train_stage4 as t4
importlib.reload(t4)

attn_tag = ("cbam" if USE_CHANNEL_ATTN and USE_TEMPORAL_ATTN
            else ("chan" if USE_CHANNEL_ATTN
                  else ("temp" if USE_TEMPORAL_ATTN else "none")))
RUN_ID_EFFECTIVE = RUN_ID or f"{SUBSET.lower()}_{attn_tag}_seed{SEED}"
OUTPUT_DIR     = f"{DRIVE_ROOT}/runs/{SUBSET}/{RUN_ID_EFFECTIVE}"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"

argv = [
    "--subset", SUBSET,
    "--seed",   str(SEED),
    "--graph-method", GRAPH_METHOD,
    "--attn-reduction", str(ATTN_REDUCTION),
    "--attn-kernel",    str(ATTN_KERNEL),
    "--gnn-hidden",     str(GNN_HIDDEN),
    "--gnn-layers",     str(GNN_LAYERS),
    "--gnn-kind",       GNN_KIND,
    "--gnn-dropout",    str(GNN_DROPOUT),
    "--gnn-pool",       GNN_POOL,
    "--run-id",         RUN_ID_EFFECTIVE,
    "--output-dir",     OUTPUT_DIR,
    "--checkpoint-dir", CHECKPOINT_DIR,
]
if USE_CHANNEL_ATTN:  argv.append("--use-channel-attn")
if USE_TEMPORAL_ATTN: argv.append("--use-temporal-attn")

for flag, val in [
    ("--recurrent-kind", RECURRENT_KIND), ("--hidden-size", HIDDEN_SIZE),
    ("--num-layers", NUM_LAYERS), ("--dropout", DROPOUT),
    ("--learning-rate", LR), ("--batch-size", BATCH_SIZE),
    ("--epochs", EPOCHS),
]:
    if val is not None:
        argv += [flag, str(val)]

print("argv:", argv)
saved = sys.argv
sys.argv = ["train_stage4"] + argv
try:
    args = t4.parse_args()
    summary = t4.run(args)
finally:
    sys.argv = saved

print(f"\n>>> test_rmse={summary['test_rmse']:.4f}  test_score={summary['test_score']:.4f}")

## Cell 8 — Inspect single-run summary + train log

In [ ]:
import json
import pandas as pd
from pathlib import Path

run_dir = Path(OUTPUT_DIR)
summary = json.loads((run_dir / "summary.json").read_text())
print(json.dumps({k: summary[k] for k in (
    "subset", "run_id", "seed", "use_gnn", "graph_method",
    "use_channel_attn", "use_temporal_attn",
    "parameter_count", "best_epoch", "best_metric",
    "test_rmse", "test_score",
)}, indent=2))

log = pd.read_csv(run_dir / "train_log.csv")
log.tail(10)

## Cell 9 — Batch: 4 subsets × 3 attention cells × 3 seeds

Stage 4 main ablation. 36 runs total. `SKIP_EXISTING = True` allows
resuming after disconnect — already-completed runs are loaded from Drive
rather than retrained.

In [ ]:
import json, sys, importlib
from pathlib import Path
from statistics import mean, stdev
import stage4.train_stage4 as t4
importlib.reload(t4)

SEEDS         = [7, 42, 123]                      # match Stage 2/3 finalists
BATCH_GRAPH   = "physical"                        # Stage 3 winner
BATCH_EPOCHS  = {"FD001": 150, "FD002": 150,
                  "FD003": 250, "FD004": 150}
SKIP_EXISTING = True

# (cell_label, use_channel, use_temporal)
CELLS = [
    ("channel_only",  True,  False),
    ("temporal_only", False, True),
    ("cbam_full",     True,  True),
]
SUBSETS = ["FD001", "FD002", "FD003", "FD004"]

results = []
for subset in SUBSETS:
    for cell_label, use_c, use_t in CELLS:
        for seed in SEEDS:
            run_id       = f"{subset.lower()}_{cell_label}_seed{seed}"
            out_dir      = f"{DRIVE_ROOT}/runs/{subset}/{run_id}"
            ckpt_dir     = f"{out_dir}/checkpoints"
            summary_path = Path(out_dir) / "summary.json"

            if SKIP_EXISTING and summary_path.exists():
                print(f"[skip] {run_id}")
                s = json.loads(summary_path.read_text())
            else:
                argv = [
                    "--subset", subset,
                    "--seed",   str(seed),
                    "--graph-method", BATCH_GRAPH,
                    "--epochs", str(BATCH_EPOCHS[subset]),
                    "--run-id", run_id,
                    "--output-dir",     out_dir,
                    "--checkpoint-dir", ckpt_dir,
                ]
                if use_c: argv.append("--use-channel-attn")
                if use_t: argv.append("--use-temporal-attn")
                print(f"\n=== {subset} | {cell_label} | seed={seed} ===")
                saved = sys.argv
                sys.argv = ["train_stage4"] + argv
                try:
                    args = t4.parse_args()
                    s    = t4.run(args)
                finally:
                    sys.argv = saved

            results.append({
                "subset": subset, "cell": cell_label, "seed": seed,
                "test_rmse": s["test_rmse"], "test_score": s["test_score"],
                "best_epoch": s.get("best_epoch"),
                "params": s.get("parameter_count"),
                "train_seconds": s.get("total_train_seconds"),
            })

# ---- aggregate by (subset, cell) over seeds ----
def _mean_std(xs):
    xs = list(xs)
    if not xs:        return (float("nan"), float("nan"))
    if len(xs) == 1:  return (xs[0], 0.0)
    return (mean(xs), stdev(xs))

agg = []
for subset in SUBSETS:
    for cell_label, _, _ in CELLS:
        rs = [r for r in results if r["subset"] == subset and r["cell"] == cell_label]
        m_r, s_r = _mean_std([r["test_rmse"]  for r in rs])
        m_s, s_s = _mean_std([r["test_score"] for r in rs])
        agg.append({
            "subset": subset, "cell": cell_label, "n_seeds": len(rs),
            "rmse_mean": m_r, "rmse_std": s_r,
            "score_mean": m_s, "score_std": s_s,
        })

print("\n" + "=" * 86)
print("STAGE 4 BATCH SUMMARY  (mean \u00b1 std across seeds {7, 42, 123})")
print("=" * 86)
header = f"{'subset':<8}{'cell':<16}{'test_rmse  (mean \u00b1 std)':>30}{'test_score  (mean \u00b1 std)':>32}"
print(header)
print("-" * len(header))
for subset in SUBSETS:
    for cell_label, _, _ in CELLS:
        a = next(x for x in agg if x["subset"] == subset and x["cell"] == cell_label)
        cell_r = f"{a['rmse_mean']:7.4f} \u00b1 {a['rmse_std']:6.4f}"
        cell_s = f"{a['score_mean']:9.2f} \u00b1 {a['score_std']:7.2f}"
        print(f"{subset:<8}{cell_label:<16}{cell_r:>30}{cell_s:>32}")
    print()

out = Path(DRIVE_ROOT) / "batch_summary.json"
out.write_text(json.dumps({"runs": results, "aggregate": agg}, indent=2))
print(f"Saved: {out}")
print(f"Total runs: {len(results)} / {len(SUBSETS) * len(CELLS) * len(SEEDS)}")

## Cell 10 — Compare against Stage 3 baselines

Pulls Stage 3 finalist numbers from `stage3/docs/stage3_analysis.md`
(hardcoded here for the comparison table) and stacks each subset's three
Stage 4 attention cells beside its Stage 3 baseline.

Stage 3 baselines used:
- FD001 → no-GNN (pure GRU)
- FD002/FD003/FD004 → GNN-physical

In [ ]:
import json
from pathlib import Path

STAGE3_BASELINE = {
    "FD001": {"rmse":  6.165, "rmse_std": 0.219, "score":   55.72, "score_std":   3.66, "note": "no-gnn (Stage 3 finalist)"},
    "FD002": {"rmse": 13.705, "rmse_std": 0.188, "score": 1010.90, "score_std": 333.90, "note": "gnn-physical (Stage 3 finalist)"},
    "FD003": {"rmse":  5.181, "rmse_std": 0.414, "score":   45.28, "score_std":   5.47, "note": "gnn-physical (Stage 3 finalist)"},
    "FD004": {"rmse": 15.900, "rmse_std": 0.585, "score": 1020.29, "score_std": 102.57, "note": "gnn-physical (Stage 3 finalist)"},
}

batch = json.loads((Path(DRIVE_ROOT) / "batch_summary.json").read_text())
agg = batch["aggregate"]

print(f"{'subset':<8}{'variant':<22}{'rmse (mean \u00b1 std)':>28}{'score (mean \u00b1 std)':>28}")
print("-" * 86)
for subset in ["FD001", "FD002", "FD003", "FD004"]:
    b = STAGE3_BASELINE[subset]
    print(f"{subset:<8}{'stage3 baseline':<22}{b['rmse']:7.4f} \u00b1 {b['rmse_std']:6.4f}     {b['score']:9.2f} \u00b1 {b['score_std']:7.2f}")
    for cell_label in ["channel_only", "temporal_only", "cbam_full"]:
        a = next(x for x in agg if x["subset"] == subset and x["cell"] == cell_label)
        print(f"{'':<8}{'stage4 ' + cell_label:<22}{a['rmse_mean']:7.4f} \u00b1 {a['rmse_std']:6.4f}     {a['score_mean']:9.2f} \u00b1 {a['score_std']:7.2f}")
    print()